In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import sys
import pandas as pd

from scipy.interpolate import interp1d


import socket
import struct
import time

In [2]:
def defineCarrierArray(numberCarriers, numberActive, numberPilots):
    global numberData
    numberData = numberActive - numberPilots - 1
    
    # Initialize all carriers as 0
    carrierArray = np.zeros(numberCarriers)
    
    # Active carrier indices
    activeCarriers = np.arange(numberCarriers//2 - numberActive//2,
                               numberCarriers//2 + numberActive//2)
    
    # Compute pilot spacing
    pilot_step = numberActive // numberPilots
    
    # Select active carriers
    carrierArray[activeCarriers] = 1
    
    # Boolean mask for pilots
    pilot_mask = (activeCarriers % pilot_step) == (pilot_step / 2)
    
    # Assign pilot tones
    carrierArray[activeCarriers[pilot_mask]] = 1.5
    
    # Force DC carrier (center) to zero
    carrierArray[numberCarriers // 2] = -1
    
    #print(activeCarriers, len(activeCarriers))
    #print(carrierArray.tolist(), len(carrierArray))

    return carrierArray


# numberCarriers = 128
# numberActive = 64
# numberPilots = 4
# carrierArray = defineCarrierArray(numberCarriers, numberActive, numberPilots)


In [3]:
#plot carrier diagram

def carrierPlot(carrierArray):
    x = np.arange(len(carrierArray))
    values = np.array(carrierArray)
    
    plt.figure(figsize=(18, 6))
    
    # Plot null carriers in black
    zeros = (values == 0 or values == -1)
    plt.stem(x[zeros], values[zeros], linefmt='k-', markerfmt='ko', basefmt=' ', label='Null Carrier')
    
    # Plot data carriers in blue
    ones = (values == 1)
    plt.stem(x[ones], values[ones], linefmt='b-', markerfmt='bo', basefmt=' ', label='Data Carrier')
    
    # Plot pilot carriers in red
    one_point_five = (values == 1.5)
    plt.stem(x[one_point_five], values[one_point_five], linefmt='r-', markerfmt='ro', basefmt=' ', label='Pilot Carrier')
    
    plt.title('Carrier Plot')
    plt.legend()
    plt.grid(True)
    plt.show()

#carrierPlot(carrierArray)

In [4]:
#generate random message bits
def generateRandomMessage(length):
    """
    Simulates the Data Source by generating a frame of random bytes and 
    converting them into a serial bitstream.
    
    This mimics the behavior of a network layer payload being passed 
    down to the Physical Layer (PHY).
    
    Parameters:
    length (int): Number of bytes to generate for the message frame.
    
    Returns:
    frame (np.uint8): The original byte-array for verification/BER calculation.
    frameBits (np.uint8): The expanded bit-array ready for symbol mapping.
    """
    # 1. Byte Generation: Generate 'length' random bytes (0-255)
    # This represents the raw information before any FEC or line coding.
    frame = np.random.randint(0, 256, length, dtype=np.uint8) 
    
    # 2. Bit Unpacking: Convert bytes into a continuous stream of bits.
    # np.unpackbits turns each uint8 into 8 separate bits (Big-Endian by default).
    # This bitstream will be mapped to M-QAM symbols in the next stage.
    frameBits = np.unpackbits(frame) 
    
    return frame, frameBits

#frame, frameBits = generateRandomMessage(1024)

In [5]:
#create BPSK or QPSK frame from bits

def bpskModulation(frameBits):
    """
    Maps bits to BPSK symbols (-1, +1).
    Used for high-robustness scenarios or signaling/headers.
    """
    # 1. Bipolar Mapping: 0 -> -1, 1 -> +1
    frameBPSK = 2 * frameBits.astype(int) - 1
    
    # 2. Padding: Ensure the bitstream fits perfectly into OFDM symbols
    # 'numberData' is the count of active data subcarriers per OFDM symbol
    samplePadding = numberData - (len(frameBPSK) % numberData)
    pad = np.zeros(samplePadding)
    frameBPSK = np.concatenate((frameBPSK, pad))

    # 3. Serial-to-Parallel (S/P): Reshape into a matrix of (Symbols, Carriers)
    frameBPSK = frameBPSK.reshape(int(len(frameBPSK) / numberData), numberData)

    return frameBPSK

def qpskModulation(frameBits):
    """
    Maps bit-pairs to QPSK complex symbols using Gray Coding.
    Doubles the spectral efficiency (2 bits/symbol) to reach higher data rates.
    """
    # 1. Padding: Ensure total bits are a multiple of (2 * numberData)
    samplePadding = (2 * numberData) - (len(frameBits) % (2 * numberData))
    pad = np.zeros(samplePadding, dtype=int)
    bits = np.concatenate((frameBits.astype(int), pad))
    
    # 2. Reshape into bit-pairs for M-ary mapping (M=4)
    bit_pairs = bits.reshape(-1, 2)
    
    # 3. Gray-coded QPSK mapping: Ensures only 1 bit changes between adjacent symbols,
    # minimizing Bit Error Rate (BER) for a given Symbol Error Rate (SER).
    # Normalization by sqrt(2) ensures unit average power.
    mapping = {
        (0, 0):  1 + 1j,
        (0, 1):  1 - 1j,
        (1, 1): -1 - 1j,
        (1, 0): -1 + 1j,
    }
    
    # 4. Vectorized Mapping & Normalization
    frameQPSK = np.array([mapping[tuple(b)] for b in bit_pairs]) / np.sqrt(2)
    
    # 5. Serial-to-Parallel (S/P): Organize into OFDM frequency-domain frames
    frameQPSK = frameQPSK.reshape(int(len(frameQPSK) / numberData), numberData)
    
    return frameQPSK

def qam16Modulation(frameBits):
    """
    Maps 4-bit groups to 16-QAM complex symbols using Gray Coding.
    Increases spectral efficiency to 4 bits/symbol.
    """
    # 1. Padding: Ensure total bits are a multiple of (4 * numberData)
    # 16-QAM carries 4 bits per symbol
    bits_per_symbol = 4
    remainder = len(frameBits) % (bits_per_symbol * numberData)
    
    if remainder != 0:
        samplePadding = (bits_per_symbol * numberData) - remainder
        pad = np.zeros(samplePadding, dtype=int)
        bits = np.concatenate((frameBits.astype(int), pad))
    else:
        bits = frameBits.astype(int)
    
    # 2. Reshape into 4-bit groups
    bit_groups = bits.reshape(-1, bits_per_symbol)
    
    # 3. Gray-coded 16-QAM mapping
    # The mapping follows a grid where adjacent symbols only differ by 1 bit.
    # Logic: First two bits define Real (I), last two bits define Imaginary (Q).
    mapping = {
        (0,0,0,0):  3 + 3j, (0,0,0,1):  3 + 1j, (0,0,1,1):  3 - 1j, (0,0,1,0):  3 - 3j,
        (0,1,0,0):  1 + 3j, (0,1,0,1):  1 + 1j, (0,1,1,1):  1 - 1j, (0,1,1,0):  1 - 3j,
        (1,1,0,0): -1 + 3j, (1,1,0,1): -1 + 1j, (1,1,1,1): -1 - 1j, (1,1,1,0): -1 - 3j,
        (1,0,0,0): -3 + 3j, (1,0,0,1): -3 + 1j, (1,0,1,1): -3 - 1j, (1,0,1,0): -3 - 3j
    }
    
    # 4. Vectorized Mapping & Normalization
    # Normalization factor for 16-QAM is sqrt(10) to maintain unit average power
    norm_factor = np.sqrt(10)
    frameQAM = np.array([mapping[tuple(b)] for b in bit_groups]) / norm_factor
    
    # 5. Serial-to-Parallel (S/P): Organize into OFDM frequency-domain frames
    frameQAM = frameQAM.reshape(-1, numberData)
    
    return frameQAM

#frameBPSK = bpskModulation(frameBits)
#frameQPSK = qpskModulation(frameBits)
#print(frameBPSK, frameBPSK.shape)
#print(frameQPSK, frameQPSK.shape)

#dataBurst = frameQPSK
#dataBurst = frameBPSK


In [6]:
def arrangeFrequencyBurst(carrierArray, dataBurst, pilotCarrier, numberCarriers):
    """
    Maps modulated data and pilot symbols onto their assigned subcarriers.
    
    Parameters:
    carrierArray (np.array): The subcarrier map (0=Guard, 1=Data, 1.5=Pilot).
    dataBurst (np.array): Matrix of modulated symbols (BPSK/QPSK).
    pilotCarrier (complex): The complex value/sequence for pilot tones.
    numberCarriers (int): Total FFT size (N).
    
    Returns:
    freqBurst (np.complex): Frequency-domain OFDM symbols ready for IFFT.
    """
    # 1. Identify indices for subcarrier types using the carrierArray map
    dataMask  = (carrierArray == 1)
    pilotMask = (carrierArray == 1.5)
    
    numSlots = len(dataBurst) # Number of OFDM symbols in this burst
    
    # 2. Pre-allocate the frequency matrix
    # Shape: (Number of symbols, N/2 + 1) for use with irfft
    freqBurst = np.zeros((numSlots, numberCarriers), dtype=complex)
    
    # 3. Vectorized Mapping: Insert payload and pilots into the grid
    # This aligns with the 'numberData' calculation from earlier steps
    freqBurst[:, dataMask] = dataBurst
    freqBurst[:, pilotMask] = pilotCarrier

    return freqBurst

def freqToTime(freqBurst):
    """
    Transforms the signal from Frequency Domain to Time Domain.
    
    Uses Inverse Real Fast Fourier Transform (irfft) to generate a
    real-valued time signal, as required for Intensity Modulation (IM) 
    in VLC systems (Adiono et al., 2017).
    """
    # IFFT operation across the subcarrier axis (axis 1)
    # The irfft ensures Hermitian symmetry is maintained, producing a
    # real-valued output suitable for the Raspberry Pi's DAC/PWM output.
    timeBurst = np.fft.irfft(freqBurst, axis=1)
    
    return timeBurst

#pilotCarrier = 1
#freqBurst = arrangeFrequencyBurst(carrierArray, dataBurst, pilotCarrier)
#timeBurst = freqToTime(freqBurst)

In [7]:
#add cyclic prefix and flatten
# Add cyclic prefix and flatten
def addCyclicPrefix(cpLength, timeBurst):
    """
    Prepends a Cyclic Prefix (CP) to each OFDM symbol.
    
    In standard OFDM, the CP is placed at the beginning of the symbol. 
    This acts as a buffer against Inter-Symbol Interference (ISI) and 
    allows the linear convolution of the channel to be treated as 
    circular convolution at the receiver.
    
    Parameters:
    cpLength (int): Number of samples copied from the end to the start.
    timeBurst (np.array): Matrix of time-domain symbols (Symbols x Samples).
    
    Returns:
    OFDMSignal (np.array): Flattened time-domain signal.
    """
    # 1. Take the last 'cpLength' samples of each symbol
    # These samples will "wrap around" to the front.
    cyclicPrefix = timeBurst[:, -cpLength:] 
    
    # 2. Prepend the CP (Standard IEEE 802.11/802.15.7 approach)
    # The prefix now correctly precedes the symbol data.
    OFDMSignal = np.hstack([cyclicPrefix, timeBurst]) 
    
    # 3. Serialize the matrix into a 1D waveform
    # Shape transition: (num_symbols, samples + cp) -> (total_samples,)
    OFDMSignal = OFDMSignal.flatten()
    
    return OFDMSignal

def plotOFDMSignal(data):
    plt.figure(figsize=(64, 4))  # Wide plot
    plt.plot(data, linewidth=0.8)
    plt.title("OFDM signal")
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    print(data.min(), data.max())

#cpLength = 32
#OFDMSignal = addCyclicPrefix(cpLength, timeBurst)
#plotOFDMSignal(OFDMSignal)

In [8]:
#normalize and scale for 12-bit values.
def normalizeForDAC(OFDMSignal, DACResolution=12, clipping_factor=3.5):
    """
    Advanced Normalization using Statistical Clipping.
    
    Instead of Min-Max, we use a clipping factor (K) to map the signal 
    based on its standard deviation. This maximizes the Signal-to-Quantization 
    Noise Ratio (SQNR) for the bulk of the data.
    
    Parameters:
    OFDMSignal (np.array): Bipolar OFDM signal (mean should be ~0).
    DACResolution (int): 12 for the DAC7811.
    clipping_factor (float): K-factor. Usually between 3 and 4. 
                             K=3.5 covers 99.95% of Gaussian peaks.
    """
    DACMax = (2 ** DACResolution) - 1
    
    # 1. Calculate Statistics
    mu = np.mean(OFDMSignal)
    sigma = np.std(OFDMSignal)
    
    # 2. Define Clipping Thresholds
    # We define the 'useful' range as mu +/- (K * sigma)
    v_min = mu - (clipping_factor * sigma)
    v_max = mu + (clipping_factor * sigma)
    
    # 3. Apply Clipping (Hard Limiting)
    # This prevents the DAC from wrapping around if a peak exceeds the range
    clipped_signal = np.clip(OFDMSignal, v_min, v_max)
    
    # 4. Normalize to [0, 1] based on CLIPPED range, not absolute range
    normalized = (clipped_signal - v_min) / (v_max - v_min)
    
    # 5. Quantize to 12-bit integers
    DACSignal = np.round(DACMax * normalized).astype(np.int16) # Use int16 for 12-bit
    
    return DACSignal

#DACSignal = normalizeForDAC(OFDMSignal, 12)
#plotOFDMSignal(DACSignal)

In [9]:
def create_real_ofdm_preamble(fft_size=128, dac_scale=4096, target_rms=0.15):
    """
    Generates a Schmidl & Cox preamble with Statistical Guardbanding.
    
    We use a target_rms that ensures the signal stays within [-1, 1] 
    for 99.999% of samples, preventing integer overflow in the 12-bit DAC.
    """
    N = fft_size
    N_half = N // 2
    
    # 1. Frequency Domain Generation
    X_base = np.random.choice([1, -1], size=N_half)
    X = np.zeros(N, dtype=np.complex128)
    X[1::2] = X_base # Map to Odd subcarriers
    X[N-1::-2] = np.conjugate(X[1::2]) # Hermitian Symmetry
    X[0] = 0.0 # DC Null
    
    # 2. Time Domain Transformation
    # Resulting [P, P] structure has zero mean and Gaussian-like distribution
    preamble_td = np.fft.ifft(X, n=N).real 
    preamble_td_full = np.concatenate([preamble_td, preamble_td])
    
    # 3. STATISTICAL SCALING (The "Pro" Fix)
    # We force the RMS such that the 5-sigma peak matches the unit range.
    # K=5.0 is the 'Statistical Improbability' threshold.
    K = 5.0
    current_rms = np.sqrt(np.mean(preamble_td_full**2))
    
    # Scale signal so that K * sigma = 1.0 (Unit Range)
    preamble_td_full = preamble_td_full * (1.0 / (K * current_rms))
    
    # 4. SAFETY CLIP & DAC MAPPING
    # Even if a 6-sigma event occurs, np.clip prevents bit-wrapping/overflow
    preamble_clipped = np.clip(preamble_td_full, -1.0, 1.0)
    
    # Map [-1, 1] to [0, 4095] for the 12-bit DAC7811
    preamble_dac = ((preamble_clipped + 1) * (dac_scale / 2)).astype(np.int16)

    return preamble_dac

# --- USAGE (Example) ---

#preamble = create_real_ofdm_preamble_corrected()
#DACPreamble = np.concatenate((preamble, DACSignal, preamble))
#print(preamble.shape, DACSignal.shape, DACPreamble.shape)

#DACSignal_quantized = np.round(DACPreamble).astype(np.uint16)
#print(DACSignal_quantized)
#plotOFDMSignal(DACSignal_quantized)
#DACSignal_quantized.tofile("data.bin")

## Channel

In [10]:
import numpy as np

def simulate_vlc_link(tx_samples, distance=5.0, angle_deg=20, gain_db=10):
    """
    Simulates the full physical link: Digital -> Photons -> Noise -> Digital.
    """
    # --- 1. CONSTANTS ---
    q = 1.602e-19           # Electron charge
    k_B = 1.38e-23          # Boltzmann constant
    T_k = 298               # Room Temp
    B = 50000               # 50kHz Signal Bandwidth
    I2 = 0.562              # Noise bandwidth factor
    
    # LED/PD Specifics
    P_total = 0.5           # Total Optical Power of LED (Watts) @ 300mA
    m = 1                   # Lambertian order (m=1 for standard 60-deg half-angle)
    A_rx = 13e-6            # Effective area of PDA36A2 sensor (m^2)
    R = 0.45                # Responsivity (A/W)
    I_amb = 10e-3         # Ambient light current (10uA)
    
    # Thorlabs TIA Gain Map (V/A)
    gain_map = {0: 1.51e3, 10: 4.75e3, 20: 1.51e4, 30: 4.75e4}
    G_tia = gain_map[gain_db]
    V_fs = 1.0              # USRP/Scope Full Scale Range (1V)

    # --- 2. LAMBERTIAN LoS PROPAGATION ---
    # Calculate Channel Gain (H)
    # H = (m+1)*A / (2*pi*d^2) * cos^m(phi)
    phi = np.radians(angle_deg)
    H_los = ((m + 1) * A_rx / (2 * np.pi * distance**2)) * np.cos(phi)**m
    
    # --- 3. TRANSMITTER: SAMPLES TO PHYSICAL CURRENT ---
    # Map 12-bit to instantaneous optical power, then to received current
    # We apply the channel gain here
    i_rx_signal = (tx_samples.astype(float) / 4095.0) * P_total * H_los * R

    # --- 4. NOISE MODELING (Current Domain) ---
    # Shot Noise (Includes signal, bias, and ambient)
    # Note: Mean current is used for the shot noise variance
    sigma_shot_sq = 2 * q * (np.mean(i_rx_signal) + I_amb) * B * I2
    
    # Thermal Noise (Based on Thorlabs TIA Gain as Load Resistance)
    sigma_thermal_sq = (4 * k_B * T_k / G_tia) * B * I2
    
    sigma_total = np.sqrt(sigma_shot_sq + sigma_thermal_sq)
    noise = np.random.normal(0, sigma_total, i_rx_signal.shape)
    
    i_rx_noisy = i_rx_signal + noise

    # --- 5. RECEIVER: CURRENT TO 12-BIT SAMPLES ---
    # Convert Current back to Voltage using TIA Gain
    v_out = i_rx_noisy * G_tia
    
    # Quantize to 12-bit (mapping 0-V_fs to 0-4095)
    rx_samples = np.clip(v_out / V_fs, 0, 1)
    rx_samples = np.round(rx_samples * 4095).astype(np.uint16)
    
    return rx_samples, sigma_total, H_los

# Example Run:
# samples_tx = your_ofdm_buffer
# samples_rx, noise_floor, path_loss = simulate_vlc_link(samples_tx, distance=1.5, angle_deg=15)

## Reception

In [11]:
from scipy.interpolate import interp1d

def simulate_timing_offset(x, t_offset=0.0, wrap=True):
    """
    Simulates the misalignment between the TX and RX clocks.
    
    This function introduces both integer-sample delays (coarse) and 
    fractional-sample delays (fine), mimicking the sampling jitter 
    inherent in low-cost hardware like the Raspberry Pi.
    
    Parameters:
    x (np.array): The received signal vector.
    t_offset (float): The offset in samples (e.g., 2.5 samples).
    wrap (bool): If True, signal wraps around (useful for testing FFT properties).
                If False, zero-padding is used (more realistic for a burst).
    """
    N = len(x)

    # 1. Decomposition of the offset
    # Separate the offset into a discrete shift and a sub-sample interpolation
    int_offset = int(np.floor(t_offset))
    frac_offset = t_offset - int_offset

    # 2. Integer Timing Offset (Coarse Shift)
    if wrap:
        # np.roll is efficient for periodic signals (idealized case)
        x_shifted = np.roll(x, int_offset)
    else:
        # Concatenation is used for a 'one-shot' burst transmission
        if int_offset >= 0:
            x_shifted = np.concatenate([np.zeros(int_offset), x[:N-int_offset]])
        else:
            n_abs = abs(int_offset)
            x_shifted = np.concatenate([x[n_abs:], np.zeros(n_abs)])

    # 3. Fractional Timing Offset (Fine Shift via Linear Interpolation)
    # This simulates the Inter-Symbol Interference (ISI) caused by
    # the ADC sampling at the 'wrong' instant.
    if abs(frac_offset) > 1e-6:
        n = np.arange(N)
        # We create a continuous representation of the discrete signal
        interp = interp1d(n, x_shifted, kind="linear", fill_value=0, bounds_error=False)
        
        # We sample the continuous signal at the shifted time indices
        n_new = n - frac_offset
        x_shifted = interp(n_new)

    return x_shifted

In [12]:


# -----------------------------
# 2. Fractional shift using FFT
# -----------------------------
def fractional_shift_fft(x, frac):
    """
    Fractional sample delay using FFT.
    Positive frac = delay.
    """
    N = len(x)
    X = np.fft.fft(x)
    n = np.arange(N)
    X_shifted = X * np.exp(-2j * np.pi * frac * n / N)
    y = np.fft.ifft(X_shifted)
    return np.real(y)

def simulate_timing_offset(x, t_offset):
    int_offset = int(np.floor(t_offset))
    frac_offset = t_offset - int_offset

    # Integer shift
    x_int = np.roll(x, int_offset)

    # Fractional shift
    if abs(frac_offset) > 1e-6:
        x_shifted = fractional_shift_fft(x_int, frac_offset)
    else:
        x_shifted = x_int
    return x_shifted

# -----------------------------
# 4. Integer timing recovery
# -----------------------------
def recover_integer_timing(r, preamble, search_window=None):
    if search_window is None:
        search_window = len(r)
    corr = np.correlate(r[:search_window], preamble, mode='valid')
    start_idx = np.argmax(corr)
    return start_idx, corr

# -----------------------------
# 5. Robust normalization
# -----------------------------
def normalize_dac(y, dac_min=0, dac_max=4096, clip_percentile=0.1):
    """
    Normalize signal to DAC range while ignoring extreme outliers.
    clip_percentile: percentage of samples at top/bottom to ignore
    """
    low = np.percentile(y, clip_percentile)
    high = np.percentile(y, 100 - clip_percentile)
    y_clipped = np.clip(y, low, high)
    y_norm = (y_clipped - np.min(y_clipped)) / (np.max(y_clipped) - np.min(y_clipped))
    y_norm = y_norm * (dac_max - dac_min) + dac_min
    return y_norm

# -----------------------------
# 6. Fractional timing recovery with normalization
# -----------------------------
def apply_fractional_delay_normalized(rx, frac_offset, dac_min=0, dac_max=4096):
    y = fractional_shift_fft(rx, -frac_offset)
    y = normalize_dac(y, dac_min, dac_max, clip_percentile=0.1)
    return y

# -----------------------------
# 7. Similarity metrics
# -----------------------------
def mse(x, y):
    return np.mean((x - y)**2)

def nmse(x, y):
    return np.sum((x - y)**2) / np.sum(x**2)

def snr_db_physical(signal_at_pd, sigma_noise):
    """
    Robust SNR calculation that avoids the 'Ambiguous Truth Value' error.
    """
    # 1. Ensure we are dealing with a scalar power value
    # np.var() returns a single float, which is safe.
    P_signal = np.var(signal_at_pd)
    
    # 2. Ensure noise power is a scalar
    # If sigma_noise is an array, take its mean or the first element
    if isinstance(sigma_noise, np.ndarray):
        P_noise = np.mean(sigma_noise**2)
    else:
        P_noise = float(sigma_noise)**2
    
    # 3. Prevent Division by Zero
    eps = 1e-20
    if P_noise <= eps:
        return 100.0 # High cap for noiseless signal
        
    # 4. Calculate SNR
    ratio = P_signal / P_noise
    
    # If P_signal is 0, ratio is 0, log10(0) is -inf. 
    # We check if ratio is greater than 0 before log.
    if ratio > 0:
        snr_value = 10 * np.log10(ratio)
    else:
        snr_value = -99.0 # Signal is effectively dead
        
    return snr_value



# -----------------------------
# Plot signals
# -----------------------------
def plotSyncMetrics(dataOriginal, dataRecovered):
    plt.figure(figsize=(64,6))
    plt.subplot(3,1,1)
    plt.plot(DACPreamble, label="Original TX signal")
    plt.legend()
    plt.subplot(3,1,2)
    plt.plot(rx_aligned, label="Received with timing offset")
    plt.legend()
    plt.subplot(3,1,3)
    plt.plot(rx_resynced, label="Resynchronized & normalized signal")
    plt.legend()
    plt.tight_layout()
    plt.show()




In [13]:
# --- Automated Payload Extraction (Based on successful full-frame sync) ---
def removePreamble(preamble, DACSignal, rx_resynced):
    """
    Strips the synchronization preambles from the synchronized received frame.
    
    This function assumes that the 'rx_resynced' signal has already been 
    corrected for timing and frequency offsets using the Schmidl & Cox metric.
    
    Parameters:
    preamble (np.array): The reference preamble used at the TX.
    DACSignal (np.array): The original payload signal before frame assembly.
    rx_resynced (np.array): The captured signal after time-alignment.
    """
    # 1. Define Segment Lengths
    # L_preamble_start: Length of the S&C synchronization overhead (e.g., 2N samples)
    L_preamble_start = len(preamble) 
    
    # L_payload_block: Total samples of the DCO-OFDM payload (including Cyclic Prefixes)
    L_payload_block = len(DACSignal) 
    
    # 2. Determine Slice Indices
    # Because 'rx_resynced' is already aligned to the first sample of the preamble (index 0),
    # the payload begins exactly after the preamble length.
    payload_start_index = L_preamble_start
    payload_end_index = L_preamble_start + L_payload_block
    
    # 3. Slice and extract the payload
    # This removes both the start preamble and any trailing signals (like an end preamble)
    rx_payload_resynced_only = rx_resynced[payload_start_index : payload_end_index]
    
    # Thesis Note: Verification of the extracted shape confirms that the 
    # synchronization window was stable enough to capture the full burst.
    return np.array(rx_payload_resynced_only)

In [14]:
def denormalizeFromDAC(rx_payload_resynced, dac_scale=4096, clipping_factor=5.0):
    """
    Reverses the Statistical Clipping and Scaling performed at the Transmitter.
    
    This function restores the received integer samples to their floating-point 
    representation, accounting for the 12-bit DAC resolution and the K-factor 
    scaling used in the DCO-OFDM mapping.
    
    Parameters:
    rx_payload_resynced (np.array): The extracted payload integers (0-4095).
    dac_scale (int): 4096 for the 12-bit DAC7811.
    clipping_factor (float): The K value used at TX (e.g., 5.0).
    
    Returns:
    denormalized_signal (np.array): Bipolar signal centered near zero.
    """
    # 1. Reverse the DAC Integer Mapping
    # Convert [0, 4095] back to a normalized float range [0, 1]
    normalized = rx_payload_resynced.astype(float) / (dac_scale - 1)
    
    # 2. Reverse the Shift and Scale
    # TX Mapping was: (clipped_signal + 1) * (scale / 2)
    # RX Mapping: (normalized * 2) - 1
    # This centers the signal back around 0 (bipolar)
    bipolar_signal = (normalized * 2) - 1
    
    # 3. Apply Inverse K-factor Scaling (Optional but recommended)
    # While FFT-based OFDM is somewhat scale-invariant, multiplying by 
    # (1/K) restores the signal to the approximate original variance.
    # Note: Clipping distortion cannot be reversed, but the bulk signal is recovered.
    denormalized_signal = bipolar_signal / clipping_factor
    
    return denormalized_signal

In [15]:
def removeCP(OFDMRxSignal, freqBurst, cpLength):
    """
    Strips the Cyclic Prefix. 
    N = 2 * (numberCarriers - 1) = 254 for your setup.
    """
    # Time-domain samples per symbol (N + CP)
    N = 2 * (freqBurst.shape[1] - 1)
    symbol_total_len = N + cpLength
    
    num_symbols = len(OFDMRxSignal) // symbol_total_len
    # Reshape to (Symbols, N + CP)
    OFDMBlocks = OFDMRxSignal[:num_symbols * symbol_total_len].reshape(num_symbols, symbol_total_len)
    
    # Strip the first cpLength samples
    return OFDMBlocks[:, cpLength:]

In [16]:
def zfChannelEstimation(carrierArray, OFDMBlocks, pilotCarrier, modulation):
    """
    Performs Zero-Forcing equalization and hard-decision demodulation.
    Ensures the bitstream is returned as a 1D NumPy array.
    """
    # 1. FFT to Frequency Domain (Result is N/2 + 1 = 128 bins)
    RXfreqBlocks = np.fft.rfft(OFDMBlocks, axis=1)
    num_bins = RXfreqBlocks.shape[1]
    
    # Match Mask to Spectrum
    ca_reduced = carrierArray[:num_bins]
    pilotMask = (ca_reduced == 1.5)
    dataMask = (ca_reduced == 1)
    pilotIndex = np.where(pilotMask)[0]
    
    # 2. Channel Estimation
    HPilots = RXfreqBlocks[:, pilotMask] / pilotCarrier
    interp_fn = interp1d(pilotIndex, HPilots, kind="linear", axis=1, 
                         fill_value="extrapolate", assume_sorted=True)
    H_estimated = interp_fn(np.arange(num_bins))
    
    # 3. Equalization & Data Extraction
    RX_est = RXfreqBlocks / H_estimated
    RX_data = RX_est[:, dataMask]
    
# 4. Demodulation
    if modulation == "qam":
        # Re-scale from unit power back to the 16-QAM grid (-3, -1, 1, 3)
        norm_RX = RX_data * np.sqrt(10)
        real_part = np.real(norm_RX).ravel()
        imag_part = np.imag(norm_RX).ravel()
    
        # Bit 0 and 1 from Real (I) component
        b0 = (real_part < 0).astype(int)            # Left/Right
        b1 = (np.abs(real_part) < 2).astype(int)    # Inner/Outer
    
        # Bit 2 and 3 from Imaginary (Q) component
        b2 = (imag_part < 0).astype(int)            # Top/Bottom
        b3 = (np.abs(imag_part) < 2).astype(int)    # Inner/Outer
    
        # Interleave bits: [b0, b1, b2, b3, b0, b1, b2, b3...]
        rxDataBurst = np.empty(real_part.size * 4, dtype=int)
        rxDataBurst[0::4] = b0
        rxDataBurst[1::4] = b1
        rxDataBurst[2::4] = b2
        rxDataBurst[3::4] = b3
        return rxDataBurst
    
    elif modulation == "qpsk":
        bit0 = (np.real(RX_data) < 0).astype(int)
        bit1 = (np.imag(RX_data) < 0).astype(int)
        rxDataBurst = np.empty(bit0.size * 2, dtype=int)
        rxDataBurst[0::2] = bit0.ravel()
        rxDataBurst[1::2] = bit1.ravel()
        return rxDataBurst
    
    else: # bpsk
        rxDataBurst = ((np.sign(np.real(RX_data).ravel()) + 1) / 2).astype(int)
        return rxDataBurst
        

In [17]:
# Repack bits and reassemble message
def repackBits(rxDataBurst, frame):
    """
    Reassembles the recovered bitstream into byte-array format.
    
    This stage reverses the 'np.unpackbits' operation performed at the 
    transmitter. It essentially reconstructs the original application-layer 
    data from the physical-layer bitstream.
    
    Parameters:
    rxDataBurst (np.array): The recovered bitstream from the demodulator.
    frame (np.array): The original transmitted byte-array (used for slicing reference).
    
    Returns:
    rxMessage (np.uint8): The reassembled message in byte format.
    """
    # 1. Slice the bitstream
    # We slice to the exact length of the original message bits (frame.size * 8)
    # to remove any trailing padding bits added for OFDM block alignment.
    expected_bit_length = frame.size * 8
    clean_bits = rxDataBurst[:expected_bit_length]
    
    # 2. Bit Repacking
    # Converts groups of 8 bits back into a single uint8 byte.
    rxMessage = np.packbits(clean_bits)
    
    # Thesis Validation:
    # A successful transmission is defined as (rxMessage == frame).all()
    # Any discrepancies here contribute to the Bit Error Rate (BER) analysis.
    
    return rxMessage

In [18]:
def runTest(numberActive, numberPilots, modulation, distance_m, angle_deg=0, gain_db=10):
    """
    VLC System Test: Integrated Digital-Physical-Digital simulation.
    """
    # --- 1. TRANSMITTER ---
    numberCarriers, cpLength, realSampleRate = 128, 32, 357.1
    frame, frameBits = generateRandomMessage(1024)
    txBits = np.array(frameBits).astype(int)
    
    carrierArray = defineCarrierArray(numberCarriers, numberActive, numberPilots)
    
    # Select Modulation
    if modulation == "qpsk":
        dataBurst = qpskModulation(txBits)
    elif modulation == "qam":
        dataBurst = qam16Modulation(txBits)
    else:
        # Default to BPSK
        dataBurst = bpskModulation(txBits)
        
    freqBurst = arrangeFrequencyBurst(carrierArray, dataBurst, 1.0+0j, numberCarriers)
    
    # Synthesis
    time_payload = freqToTime(freqBurst)
    OFDMSignal = addCyclicPrefix(cpLength, time_payload)
    
    # Digital Samples (Representing the DDS output to the UCC27282 driver)
    DACSignal = normalizeForDAC(OFDMSignal, 12, 5.0)
    
    # --- 2. PHYSICAL CHANNEL SIMULATION ---
    # This simulates the LED -> Air -> PDA36A2 path
    # rx_digital: Noisy 12-bit samples back from the ADC
    # sigma_total: The standard deviation of the current noise (Amps)
    # h_los: The geometric gain factor (Lambertian)
    rx_digital, sigma_total, h_los = simulate_vlc_link(
        DACSignal, 
        distance=distance_m, 
        angle_deg=angle_deg, 
        gain_db=gain_db
    )

    # --- 3. SNR CALCULATION (Physical Domain) ---
    # We calculate the clean signal current power at the receiver
    p_total, r_responsivity = 0.5, 0.45 # LED Power (W) and PD Responsivity (A/W)
    i_peak = p_total * h_los * r_responsivity
    i_signal_clean = (DACSignal.astype(float) / 4095.0) * i_peak
    
    signal_power = np.mean(i_signal_clean**2)
    noise_power = sigma_total**2
    
    # Check for zero noise to avoid division by zero
    if noise_power > 0:
        snr_linear = signal_power / noise_power
        snr_db = 10 * np.log10(snr_linear)
    else:
        snr_db = 100.0 # Theoretical perfect link

    # --- 4. RECEIVER DSP ---
    # Convert quantized 12-bit samples back to floats for processing
    OFDMRxSignal = denormalizeFromDAC(rx_digital, clipping_factor=5.0)
    
    # Strip CP and Demodulate
    OFDMBlocks = removeCP(OFDMRxSignal, freqBurst, cpLength)
    rxDataBurst = zfChannelEstimation(carrierArray, OFDMBlocks, 1.0+0j, modulation)
    
    # --- 5. ERROR CALCULATION ---
    rxBits_trimmed = rxDataBurst[:len(txBits)]
    error_count = np.sum(rxBits_trimmed != txBits)
    ber = error_count / len(txBits)
    
    # Goodput calculation
    realRate = (len(txBits) * realSampleRate) / (len(DACSignal))
    if ber > 0:
        realRate *= (1 - ber)

    return ber, snr_db, realRate, h_los

In [29]:
def singleTest(n, runs, distance):
    """
    Performs a Monte Carlo simulation for a set of transmission configurations.
    
    This function aggregates multiple 'runTest' iterations to provide 
    statistically significant averages for Bit Error Rate (BER), 
    Throughput, and SNR at a specific distance.
    
    Parameters:
    n (int): Number of iterations per configuration (for statistical convergence).
    runs (list of tuples): Configurations to test, e.g., [(active_car, pilots, 'qpsk'), ...].
    distance (float): Physical distance in meters between TX and RX.
    """
    results = []
    
    for run in runs:
        data = []
        # Each 'run' defines a unique subcarrier/modulation configuration
        print(f"Testing Config: Active={run[0]}, Pilots={run[1]}, Mod={run[2]}")
        
        for i in range(n):
            # Execute the end-to-end signal chain
            data.append(runTest(run[0], run[1], run[2], distance))
        
        data = np.asarray(data)
        
        # --- Statistical Aggregation ---
        # We calculate the mean across all 'n' trials to filter out 
        # instantaneous noise spikes in the physical channel model.
        aBer      = np.average(data[:, 0]) # Average Bit Error Rate
        aSNR = np.average(data[:, 1]) # Theoretical bit rate (kbps)
        aRealRate = np.average(data[:, 2]) # Effective throughput (Goodput)
        
        print(f"{n} Iterations Completed")
        print(run)
        print(f"Avg BER: {aBer:.2e}")
        print(f"Avg Goodput: {aRealRate:.2f} kbps")
        print(f"Avg SNR: {aSNR:.2f} dB\n")
        
        # Store results for table generation or plotting
        results.append([run[0], run[1], run[2], aBer, aRealRate, aSNR, distance])
        
    return np.asarray(results)



# --- EXPERIMENTAL PARAMETER SWEEP ---
# Define the test matrix: (Active Carriers, Pilots, Modulation)
# This sweep covers low-density/high-robustness to high-density/high-speed.
runs = [
    (16, 4, "qpsk"), (32, 4, "qpsk"), (64, 4, "qpsk"), (96, 4, "qpsk"),
    (16, 8, "qpsk"), (32, 8, "qpsk"), (64, 8, "qpsk"), (96, 8, "qpsk"),
    (16, 4, "bpsk"), (32, 4, "bpsk"), (64, 4, "bpsk"), (96, 4, "bpsk"),
    (16, 8, "bpsk"), (32, 8, "bpsk"), (64, 8, "bpsk"), (96, 8, "bpsk"),
    (16, 4, "qam"), (32, 4, "qam"), (64, 16, "qam"), (96, 16, "qam"),
    (16, 8, "qam"), (32, 8, "qam"), (64, 8, "qam"), (96, 8, "qam"),
]

# Execute the Monte Carlo simulation
test = singleTest(10, runs, 1.2)

# --- DATA EXPORT FOR EXTERNAL ANALYSIS ---
# Converting the numpy results into a Pandas DataFrame for structured analysis.
# This allows for easy plotting and inclusion in the 'Análisis de Resultados' chapter.
df = pd.DataFrame(test, columns=[
    'Carriers', 'Pilots', 'Modulation', 'BER', 'Real Rate', 'SNR', 'distance'
])

# Export to CSV: This serves as the 'raw data' for your thesis appendix.
df.to_csv("data.csv", index=False, mode='a')

Testing Config: Active=16, Pilots=4, Mod=qpsk
10 Iterations Completed
(16, 4, 'qpsk')
Avg BER: 0.00e+00
Avg Goodput: 27.42 kbps
Avg SNR: 30.81 dB

Testing Config: Active=32, Pilots=4, Mod=qpsk
10 Iterations Completed
(32, 4, 'qpsk')
Avg BER: 0.00e+00
Avg Goodput: 67.29 kbps
Avg SNR: 30.81 dB

Testing Config: Active=64, Pilots=4, Mod=qpsk
10 Iterations Completed
(64, 4, 'qpsk')
Avg BER: 8.54e-05
Avg Goodput: 146.11 kbps
Avg SNR: 30.81 dB

Testing Config: Active=96, Pilots=4, Mod=qpsk
10 Iterations Completed
(96, 4, 'qpsk')
Avg BER: 2.60e-03
Avg Goodput: 221.78 kbps
Avg SNR: 30.81 dB

Testing Config: Active=16, Pilots=8, Mod=qpsk
10 Iterations Completed
(16, 8, 'qpsk')
Avg BER: 0.00e+00
Avg Goodput: 17.45 kbps
Avg SNR: 30.81 dB

Testing Config: Active=32, Pilots=8, Mod=qpsk
10 Iterations Completed
(32, 8, 'qpsk')
Avg BER: 0.00e+00
Avg Goodput: 57.14 kbps
Avg SNR: 30.81 dB

Testing Config: Active=64, Pilots=8, Mod=qpsk
10 Iterations Completed
(64, 8, 'qpsk')
Avg BER: 6.10e-05
Avg Goodput: